In [ ]:
import cv2

import torch
import torch.nn as nn

from torchvision import transforms

from PIL import Image

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

classnames = [
    "bottle",
    "pen",
    "flashlight"
]

print(device)

cuda


(null): No such file or directory


In [3]:
class Letterbox:
    def __init__(self, size=256, fill=0):
        self.size = size
        self.fill = fill

    def __call__(self, img: Image.Image):

        w, h = img.size

        scale = min(self.size / w, self.size / h)

        new_w = int(w * scale)
        new_h = int(h * scale)

        img = img.resize((new_w, new_h), Image.BILINEAR)

        new_img = Image.new(
            "RGB",
            (self.size, self.size),
            (self.fill, self.fill, self.fill)
        )

        paste_x = (self.size - new_w) // 2
        paste_y = (self.size - new_h) // 2

        new_img.paste(img, (paste_x, paste_y))

        return new_img

In [4]:
transform = transforms.Compose([
    Letterbox(190),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
# CNN
class CNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.SiLU(),
            nn.Dropout(0.05),
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.SiLU(),
            nn.Dropout(0.1),
        )

        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.SiLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.15),
        )

        self.conv4 = nn.Sequential(
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.SiLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.15),
        )

        self.conv5 = nn.Sequential(
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.SiLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.15),
        )

        self.conv6 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.SiLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.15),
        )

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Sequential(
            nn.Linear(128, 256),
            nn.SiLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.conv5(x)
        x = self.conv6(x)

        x = self.pool(x)
        x = x.view(x.size(0), -1)

        return self.fc(x)

In [6]:
MODEL_PATH = "assets/best_model.pth"

model = CNN(num_classes=len(classnames))

model.load_state_dict(
    torch.load(MODEL_PATH, map_location=device)
)

model.to(device)

model.eval()

print("Model wczytany")

Model wczytany


In [7]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Nie można otworzyć kamerki")
else:
    print("Kamera działa")

Kamera działa


In [8]:
while True:

    ret, frame = cap.read()

    if not ret:
        break

    # OpenCV BGR -> RGB
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # PIL
    pil_image = Image.fromarray(rgb)

    # preprocessing
    input_tensor = transform(pil_image)

    # batch dimension
    input_tensor = input_tensor.unsqueeze(0)

    input_tensor = input_tensor.to(device)

    # inference
    with torch.no_grad():

        outputs = model(input_tensor)

        probs = torch.softmax(outputs, dim=1)

        confidence, predicted = torch.max(probs, 1)

    conf = confidence.item()

    if conf < 0.40:
        label = "unknown"
    else:
        label = classnames[predicted.item()]

    text = f"{label}: {conf:.2f}"

    # rysowanie tekstu
    cv2.putText(
        frame,
        text,
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )

    cv2.imshow("Object Detection", frame)

    key = cv2.waitKey(1)

    # ESC
    if key == 27:
        break

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in "/home/user/pjatk/wma/wma_projekt4/.venv/lib/python3.12/site-packages/cv2/qt/plugins"
QFontDatabase: Cannot find font directory /home/user/pjatk/wma/wma_projekt4/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/user/pjatk/wma/wma_projekt4/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/user/pjatk/wma/wma_projekt4/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/user/pjatk/wma/wma_projekt4/.venv/lib/python3.12/site

In [9]:
cap.release()

cv2.destroyAllWindows()